In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import

In [ ]:
!pip install sentence_transformers

In [ ]:
import re
import pandas as pd
import numpy as np
import random
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

## Random Seed

In [ ]:
SEED = 0

np.random.seed(SEED)
random.seed(SEED)

## Load Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/news.csv')
df.head()

,id,title,contents
0,NEWS_00000,Spanish coach facing action in race row,MADRID (AFP) - Spanish national team coach Lui...
1,NEWS_00001,Bruce Lee statue for divided city,"In Bosnia, where one man #39;s hero is often a..."
2,NEWS_00002,Only Lovers Left Alive's Tilda Swinton Talks A...,Yasmine Hamdan performs 'Hal' which she also s...
3,NEWS_00003,Macromedia contributes to eBay Stores,Macromedia has announced a special version of ...
4,NEWS_00004,Qualcomm plans to phone it in on cellular repairs,Over-the-air fixes for cell phones comes to Qu...


In [ ]:
# 제목 + 내용
df['text'] = df['title'] + ' : ' + df['contents']
df.head()

,id,title,contents,text
0,NEWS_00000,Spanish coach facing action in race row,MADRID (AFP) - Spanish national team coach Lui...,Spanish coach facing action in race row : MADR...
1,NEWS_00001,Bruce Lee statue for divided city,"In Bosnia, where one man #39;s hero is often a...","Bruce Lee statue for divided city : In Bosnia,..."
2,NEWS_00002,Only Lovers Left Alive's Tilda Swinton Talks A...,Yasmine Hamdan performs 'Hal' which she also s...,Only Lovers Left Alive's Tilda Swinton Talks A...
3,NEWS_00003,Macromedia contributes to eBay Stores,Macromedia has announced a special version of ...,Macromedia contributes to eBay Stores : Macrom...
4,NEWS_00004,Qualcomm plans to phone it in on cellular repairs,Over-the-air fixes for cell phones comes to Qu...,Qualcomm plans to phone it in on cellular repa...


## Pre-processing

In [ ]:
def preprocess_text(text):
    # URL 제거
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 해시태그 제거
    text = re.sub(r'#\w+', '', text)

    # 멘션 제거
    text = re.sub(r'@\w+', '', text)

    # 이모지 제거
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 공백 및 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip()

    # 숫자 제거
    text = re.sub(r'\d+', '', text)

    return text.lower()

In [ ]:
df['processed_text'] = df['text'].apply(preprocess_text)

## Feature Extraction

In [ ]:
# Sentence BERT 모델 로드
model = SentenceTransformer('paraphrase-distilroberta-base-v1')

# 텍스트 feature 추출
sentence_embeddings = model.encode(df['text'].tolist())

# 추출한 feature를 데이터프레임에 저장
df_embeddings = pd.DataFrame(sentence_embeddings)

## Clustering

In [ ]:
# Sentence BERT 임베딩을 사용하여 군집화 수행
kmeans = KMeans(n_clusters=6, random_state=SEED)

df['kmeans_cluster'] = kmeans.fit_predict(sentence_embeddings)

## Post-processing

### Sports: 0 -> 3

In [ ]:
df[df['kmeans_cluster'] == 0]['text'].head(3)

0     Spanish coach facing action in race row : MADR...
13    GAME DAY PREVIEW Game time: 6:00 PM : CHARLOTT...
22    College Basketball: Georgia Tech, UConn Win : ...
Name: text, dtype: object

In [ ]:
print(df['text'][1])
print(df['text'][10])
print(df['text'][16])

Bruce Lee statue for divided city : In Bosnia, where one man #39;s hero is often another man #39;s villain, some citizens have decided to honour one whom Serbs, Croats and Muslims can all look up to - the kung fu great Bruce Lee.
Harry #39;s argy-bargy : PRINCE Charles has asked Scotland Yard for an in-depth report on his son Harry #39;s trip to Argentina after reports of excessive drinking and a kidnap plot.
Fischer's Fiancee: Marriage Plans Genuine (AP) : AP - Former chess champion Bobby Fischer's announcement thathe is engaged to a Japanese woman could win him sympathy among Japanese officials and help him avoid deportation to the United States, his fiancee and one of his supporters said Tuesday.


### Tech: 1 -> 4

In [ ]:
df[df['kmeans_cluster'] == 1]['text'].head(3)

3    Macromedia contributes to eBay Stores : Macrom...
4    Qualcomm plans to phone it in on cellular repa...
5    Thomson to Back Both Blu-ray and HD-DVD : Comp...
Name: text, dtype: object

In [ ]:
df[df['kmeans_cluster'] == 1]['text']

3        Macromedia contributes to eBay Stores : Macrom...
4        Qualcomm plans to phone it in on cellular repa...
5        Thomson to Back Both Blu-ray and HD-DVD : Comp...
23       FTC Files First Lawsuit Against Spyware Concer...
27       Kmart-Sears merger about price, quality : Aver...
                               ...                        
59978    SMIC to challenge latest TSMC infringement cla...
59983    Partners Weigh In On Firefox, IE Faceoff : Sol...
59985    European Shares Tread Water (Reuters) : Reuter...
59991    Irish talk of softening schedule a little : wh...
59999    Cassini Craft Spies Saturn Moon Dione (AP) : A...
Name: text, Length: 12948, dtype: object

In [ ]:
print(df['text'][0])
print(df['text'][13])
print(df['text'][22])

### Business: 2 -> 2

In [ ]:
df[df['kmeans_cluster'] == 2]['text']

11       Kerry rolls out tax-cut plan for middle class ...
20       Deere's Color Is Green : With big tractors, bi...
50       UN Predicts Boom In Robot Labor : The use of r...
51       Oil Falls Below \$49 on Nigeria Cease-Fire : L...
61       Typhoon-Like Gusts Hit Japan; 13 Injured : TOK...
                               ...                        
59977    Lowe's Reports 18 Pct. Hike in Profits : ATLAN...
59986    Higher trade growth predicted in 2004 despite ...
59994    Chain Store Sales Rise in the Latest Week : NE...
59996    After Steep Drop, Price of Oil Rises : The fre...
59998    Albertsons on the Rebound : The No. 2 grocer r...
Name: text, Length: 6434, dtype: object

In [ ]:
df[df['kmeans_cluster'] == 2]['text'].head(3)

11    Kerry rolls out tax-cut plan for middle class ...
20    Deere's Color Is Green : With big tractors, bi...
50    UN Predicts Boom In Robot Labor : The use of r...
Name: text, dtype: object

In [ ]:
print(df['text'][2])
print(df['text'][6])
print(df['text'][7])

Only Lovers Left Alive's Tilda Swinton Talks About Almost Quitting Acting and Yasmine Hamdan Performs 'Hal' Live In NYC   (HuffPo Exclusive Videos) authors : Yasmine Hamdan performs 'Hal' which she also sings in the film during a scene when two world-weary vampires begin to heal and find a way to continue living as they remember the power and mystery of creation itself.
Time to Talk Baseball : It's time to talk about the serious risks and potential benefits of building an expensive ballpark in Washington.
Bump Stock Maker Resumes Sales One Month After Las Vegas Mass Shooting authors : Move along nothing to see here.


### World: 3 -> 5

In [ ]:
df[df['kmeans_cluster'] == 3]['text'].head(3)

1     Bruce Lee statue for divided city : In Bosnia,...
10    Harry #39;s argy-bargy : PRINCE Charles has as...
16    Fischer's Fiancee: Marriage Plans Genuine (AP)...
Name: text, dtype: object

In [ ]:
df[df['kmeans_cluster'] == 3]['text']

1        Bruce Lee statue for divided city : In Bosnia,...
10       Harry #39;s argy-bargy : PRINCE Charles has as...
16       Fischer's Fiancee: Marriage Plans Genuine (AP)...
29       Israel Kills 3 Palestinians in Big Gaza Incurs...
36       Agencies Postpone Issuing New Rules Until Afte...
                               ...                        
59979    Blair to Urge End to Trans-Atlantic Rift (Reut...
59982    Militants kill 12 in J amp;K ahead of PMs visi...
59984    The Plot Thickens: Testing European Tolerance ...
59987    Nepal Seeks to Break Rebel Siege with Air Patr...
59992    US troops on offensive in Iraq : US troops wen...
Name: text, Length: 12547, dtype: object

In [ ]:
print(df['text'][11])
print(df['text'][20])
print(df['text'][50])

### Entertainment: 4 -> 1

In [ ]:
df[df['kmeans_cluster'] == 4]['text']

2        Only Lovers Left Alive's Tilda Swinton Talks A...
6        Time to Talk Baseball : It's time to talk abou...
7        Bump Stock Maker Resumes Sales One Month After...
8        Obama Marks Anniversary Of 9/11 Attacks With M...
9        Republican Congressman Says Trump Should Apolo...
                               ...                        
59959    US Era of Dominance Is Dwindling as China Take...
59964    Russell Crowe Reaps Shocking Sum In Divorce Au...
59966    Mitt Romney Lambasts Donald Trump As A 'Phony'...
59967    Europeans in Thrall of American Culture (AP) :...
59981    Khloe Kardashian Gets A Hilarious Lesson In Po...
Name: text, Length: 13954, dtype: object

In [ ]:
df[df['kmeans_cluster'] == 4]['text'].head(3)

2    Only Lovers Left Alive's Tilda Swinton Talks A...
6    Time to Talk Baseball : It's time to talk abou...
7    Bump Stock Maker Resumes Sales One Month After...
Name: text, dtype: object

In [ ]:
print(df['text'][2])
print(df['text'][6])
print(df['text'][7])

Only Lovers Left Alive's Tilda Swinton Talks About Almost Quitting Acting and Yasmine Hamdan Performs 'Hal' Live In NYC   (HuffPo Exclusive Videos) authors : Yasmine Hamdan performs 'Hal' which she also sings in the film during a scene when two world-weary vampires begin to heal and find a way to continue living as they remember the power and mystery of creation itself.
Time to Talk Baseball : It's time to talk about the serious risks and potential benefits of building an expensive ballpark in Washington.
Bump Stock Maker Resumes Sales One Month After Las Vegas Mass Shooting authors : Move along nothing to see here.


### Politics: 5 -> 2

In [ ]:
df[df['kmeans_cluster'] == 5]['text']

18       A Fair Way to Choose Candidates for Republican...
25       Be on TOP : //www.huffingtonpost.com/entry/be-...
33       Memo To EPA Chief Pruitt : //www.huffingtonpos...
68       Satire Will Not Save Us : //www.huffingtonpost...
76       WATCH : //www.huffingtonpost.com/entry/perrish...
                               ...                        
59906                Report : //www.huffingtonpost.comhttp
59912    Joe Biden : //www.huffingtonpost.com/entry/joe...
59923    Bernie in Bush-Obama America : //www.huffingto...
59924    Women And Minorities Are Punished For Promotin...
59988    Obama To Call For More Icebreakers In The Arct...
Name: text, Length: 3224, dtype: object

In [ ]:
df[df['kmeans_cluster'] == 5]['text'].head(3)

18    A Fair Way to Choose Candidates for Republican...
25    Be on TOP : //www.huffingtonpost.com/entry/be-...
33    Memo To EPA Chief Pruitt : //www.huffingtonpos...
Name: text, dtype: object

### Mapping

In [ ]:
mapping_dict = {
    0: 3,
    1: 4,
    2: 2,
    3: 5,
    4: 1,
    5: 2
}

In [ ]:
df['mapping'] = df['kmeans_cluster'].apply(lambda x: mapping_dict[x])

## Submission

In [ ]:
sample.to_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/baseline_submit.csv', index=False)